# PlugIR on Google Colab

Runs the PlugIR experiments (ACL 2024) from this repo, with Qwen as the LLM for dialogue generation.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better).

What this notebook does:
1. Installs dependencies and clones this repo
2. Keeps cache and results in Colab (downloadable at the end) or, optionally, in your Google Drive
3. Downloads the COCO 2017 unlabeled images (~19 GB, into Colab's local disk)
4. Runs the zero-shot baseline and the PlugIR evaluation
5. *(Optional)* Generates new dialogues with Qwen and evaluates them
6. Downloads the cache and results to your computer

## 1. Setup

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [7]:
%cd /content
!git clone https://github.com/nopharshalom/ChatIR-Cornell2026 2>/dev/null || (cd ChatIR-Cornell2026 && git pull)
%cd /content/ChatIR-Cornell2026

/content
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.57 KiB | 1.57 MiB/s, done.
From https://github.com/nopharshalom/ChatIR-Cornell2026
   11c1954..e427356  main       -> origin/main
Updating 11c1954..e427356
Fast-forward
 PlugIR_Colab.ipynb | 75 +++++++++++++++++++++++++++++++++++++++++++++---------
 1 file changed, 63 insertions(+), 12 deletions(-)
/content/ChatIR-Cornell2026


In [8]:
# PyTorch and transformers come preinstalled on Colab.
# CLIP still imports pkg_resources, which newer setuptools removed.
!pip install -q "setuptools<81" ftfy regex fast-pytorch-kmeans "openai>=1.0"
!pip install -q --no-build-isolation git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


## 2. Where to keep cache and results

`cache/` holds the image-embedding cache (~50 MB, takes a while to compute) and `logs/` holds the results.

- **`USE_DRIVE = False`** (default): everything stays in Colab and is **deleted when the session ends**. Run the last cell of this notebook to download `cache/` and `logs/` to your computer. Next session, you can upload `cache/corpus_blip.pth` back to skip re-computing it.
- **`USE_DRIVE = True`**: `cache/` and `logs/` are linked to a folder in your Google Drive, so they survive automatically.

Either way, the images stay on Colab's local disk.

In [9]:
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/chatir-cornell2026'
    !mkdir -p "{SAVE_DIR}/cache" "{SAVE_DIR}/logs" "{SAVE_DIR}/dialogues"
    !rm -rf cache logs && ln -s "{SAVE_DIR}/cache" cache && ln -s "{SAVE_DIR}/logs" logs
else:
    SAVE_DIR = None
    !mkdir -p cache logs
!ls -la cache logs

Mounted at /content/drive
lrwxrwxrwx 1 root root 47 Oct 15 17:10 cache -> /content/drive/MyDrive/chatir-cornell2026/cache
lrwxrwxrwx 1 root root 46 Oct 15 17:10 logs -> /content/drive/MyDrive/chatir-cornell2026/logs


**Optional, local mode only:** if you downloaded `corpus_blip.pth` in an earlier session, upload it here to skip re-embedding the images.

In [ ]:
# from google.colab import files
# up = files.upload()               # choose corpus_blip.pth
# for name in up: os.rename(name, f'cache/{name}')

## 3. Download the COCO 2017 unlabeled images

About 19 GB. Takes roughly 10–20 minutes, and has to be redone in each new Colab session.

In [11]:
import os
if not os.path.isdir('/content/data/unlabeled2017'):
    !mkdir -p /content/data
    !wget -q --show-progress -O /content/data/unlabeled2017.zip http://images.cocodataset.org/zips/unlabeled2017.zip
    !unzip -q /content/data/unlabeled2017.zip -d /content/data && rm /content/data/unlabeled2017.zip
!ls /content/data/unlabeled2017 | wc -l

/content/data/unlab 100%[===================>]  18.74G  16.8MB/s    in 21m 28s 
123403


## 4. Evaluate

The first run embeds all 50,000 search-space images and saves them to `cache/corpus_blip.pth` (in your Drive). Later runs load that file and are much faster.

**Zero-shot baseline** (BLIP on the original VisDial dialogues):

In [12]:
!python eval.py --retriever blip --cache-corpus cache/corpus_blip.pth --data-dir /content/data \
    --output-dir logs/zeroshot_blip

2026-10-15 17:39:44,660 | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-15 17:39:44,907 | HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip-itm-large-coco/revision/main "HTTP/1.1 200 OK"
2026-10-15 17:39:45,145 | HTTP Request: HEAD https://huggingface.co/Salesforce/blip-itm-large-coco/resolve/19502f1e215844f7e48bd48473f86932486d3441/adapter_config.json "HTTP/1.1 404 Not Found"
2026-10-15 17:39:45,146 | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-15 17:39:45,387 | HTTP Request: HEAD https://huggingface.co/Salesforce/blip-itm-large-coco/resolve/19502f1e215844f7e48bd48473f86932486d3441/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-15 17:39:45,394 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/blip-itm-large-coco/19502f1e215844f7e48bd48473f86932486d3441/config.json?%2FSalesforce%2Fblip-itm-l

**PlugIR** (using the dialogues the authors generated with GPT-3.5, included in the repo):

In [13]:
!python eval.py --retriever blip --cache-corpus cache/corpus_blip.pth --data-dir /content/data \
    --queries-path dialogues/ours_final_q_n_5_recall_hitting_10_thres_low_500_recon_true_referring_true_filtering_true_select_true_reconed.json \
    --split --output-dir logs/plugir_paper_dialogues

2026-10-15 17:41:35,989 | HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip-itm-large-coco/revision/main "HTTP/1.1 200 OK"
2026-10-15 17:41:36,274 | HTTP Request: HEAD https://huggingface.co/Salesforce/blip-itm-large-coco/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-10-15 17:41:36,275 | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-15 17:41:36,514 | HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip-itm-large-coco "HTTP/1.1 200 OK"
2026-10-15 17:41:36,770 | HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip-itm-large-coco/commits/main "HTTP/1.1 200 OK"
2026-10-15 17:41:37,132 | HTTP Request: GET https://huggingface.co/api/models/Salesforce/blip-itm-large-coco/discussions?p=0 "HTTP/1.1 200 OK"
Loading weights: 100% 616/616 [00:00<00:00, 24158.61it/s]
2026-10-15 17:41:37,404 | HTTP Request: GET https://huggingface.co/api/m

## 5. *(Optional)* Generate dialogues with Qwen

Needs:
- The cache from step 4 (`cache/corpus_blip.pth`)
- A Qwen API key from QwenCloud / Alibaba Cloud Model Studio. Add it in Colab's **Secrets** panel (key icon on the left) as `DASHSCOPE_API_KEY` and turn on notebook access.
- A GPU with roughly 12 GB+ of memory (loads BLIP-2 Flan-T5-XL)

The default endpoint is the international (Singapore) one. If your key was created in another region, set `QWEN_BASE_URL` to that region's endpoint, since keys only work in their own region.

In [14]:
import os
from google.colab import userdata
os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')
os.environ['QWEN_MODEL'] = 'qwen-plus'
# os.environ['QWEN_BASE_URL'] = 'https://dashscope-intl.aliyuncs.com/compatible-mode/v1'

SecretNotFoundError: Secret DASHSCOPE_API_KEY does not exist.

Start with a small test (`E_IDX = 20` → first 20 dialogues). A full run (`E_IDX = 2064`) makes thousands of API calls, so check Qwen pricing first.

In [ ]:
E_IDX = 20

!python generate_dialog.py --q_n 5 --thres_low 500 --n_clusters 10 \
    --reconstruct --referring --filtering --select \
    --image_root /content/data --e_idx {E_IDX}

# Copy the generated dialogues to Drive (if using it) and into logs/ for the download cell
!cp dialogues/ours_final_*_0_{E_IDX}.json logs/
if SAVE_DIR:
    !cp dialogues/ours_final_*_0_{E_IDX}.json "{SAVE_DIR}/dialogues/"

**Evaluate the Qwen dialogues:**

In [ ]:
import glob
qwen_dialogues = glob.glob(f'dialogues/ours_final_*reconed_0_{E_IDX}.json')[0]
print(qwen_dialogues)

!python eval.py --retriever blip --cache-corpus cache/corpus_blip.pth --data-dir /content/data \
    --queries-path "{qwen_dialogues}" --split --output-dir logs/plugir_qwen_{E_IDX}

## 6. Download results to your computer

Zips `cache/` and `logs/` (which also holds any Qwen dialogues you generated) and downloads the zip. Do this before closing the session if `USE_DRIVE = False`.

In [ ]:
!zip -qr -FS /content/plugir_results.zip cache/ logs/
from google.colab import files
files.download('/content/plugir_results.zip')